# 📊 LMS Data Analysis Notebook

**Objective:** Analyze the cleaned LMS dataset and generate actionable insights.

**Analyses:**
1. Overall KPIs
2. Department Performance
3. Course Effectiveness
4. Monthly Trends
5. Overdue Tracking
6. Visualizations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/lms_cleaned.csv', parse_dates=['assigned_date', 'due_date', 'completion_date'])
print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## 1. Overall KPIs

In [ ]:
total = len(df)
completed = (df['status'] == 'Completed').sum()
pending = (df['status'] == 'Pending').sum()
overdue = df['is_overdue'].sum()
avg_time = df.loc[df['status'] == 'Completed', 'completion_time_days'].mean()

print(f'Total Records      : {total}')
print(f'Unique Employees   : {df["employee_id"].nunique()}')
print(f'Completed          : {completed} ({completed/total*100:.1f}%)')
print(f'Pending            : {pending} ({pending/total*100:.1f}%)')
print(f'Overdue            : {overdue} ({overdue/total*100:.1f}%)')
print(f'Avg Completion Time: {avg_time:.1f} days')

## 2. Department-wise Performance

In [ ]:
dept = df.groupby('department').agg(
    total=('status', 'count'),
    completed=('status', lambda x: (x == 'Completed').sum()),
    pending=('status', lambda x: (x == 'Pending').sum()),
    overdue=('is_overdue', 'sum'),
    avg_days=('completion_time_days', 'mean')
).reset_index()
dept['completion_rate'] = round(dept['completed'] / dept['total'] * 100, 1)
dept = dept.sort_values('completion_rate', ascending=False)
dept

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar
dept_sorted = dept.sort_values('completion_rate')
ax1.barh(dept_sorted['department'], dept_sorted['completed'], color='#22C55E', label='Completed')
ax1.barh(dept_sorted['department'], dept_sorted['pending'], left=dept_sorted['completed'], color='#F59E0B', label='Pending')
ax1.set_title('Trainings by Department')
ax1.legend()

# Completion rate
colors = ['#22C55E' if r >= 70 else '#F59E0B' if r >= 60 else '#EF4444' for r in dept_sorted['completion_rate']]
ax2.barh(dept_sorted['department'], dept_sorted['completion_rate'], color=colors)
ax2.set_title('Completion Rate (%)')
ax2.set_xlim(0, 100)
for i, v in enumerate(dept_sorted['completion_rate']):
    ax2.text(v + 1, i, f'{v}%', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Course-wise Performance

In [ ]:
course = df.groupby('course_name').agg(
    total=('status', 'count'),
    completed=('status', lambda x: (x == 'Completed').sum()),
    avg_days=('completion_time_days', 'mean')
).reset_index()
course['rate'] = round(course['completed'] / course['total'] * 100, 1)
course = course.sort_values('rate', ascending=False)
course

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
course_sorted = course.sort_values('rate')
colors = ['#22C55E' if r >= 75 else '#0891B2' if r >= 65 else '#F59E0B' if r >= 55 else '#EF4444' for r in course_sorted['rate']]
ax.barh(course_sorted['course_name'], course_sorted['rate'], color=colors)
ax.axvline(x=70, color='gray', linestyle='--', alpha=0.5, label='Target 70%')
ax.set_title('Course Completion Rates')
ax.set_xlim(0, 100)
for i, (v, n) in enumerate(zip(course_sorted['rate'], course_sorted['total'])):
    ax.text(v + 0.5, i, f'{v}% (n={n})', va='center', fontsize=10)
ax.legend()
plt.tight_layout()
plt.show()

## 4. Monthly Trends

In [ ]:
monthly = df.groupby('assigned_month').agg(
    total=('status', 'count'),
    completed=('status', lambda x: (x == 'Completed').sum()),
    pending=('status', lambda x: (x == 'Pending').sum()),
).reset_index().sort_values('assigned_month')
monthly['rate'] = round(monthly['completed'] / monthly['total'] * 100, 1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['assigned_month'], monthly['total'], 'o-', label='Total', color='#3B82F6')
ax.plot(monthly['assigned_month'], monthly['completed'], 's-', label='Completed', color='#22C55E')
ax.plot(monthly['assigned_month'], monthly['pending'], '^-', label='Pending', color='#F59E0B')
ax.set_title('Monthly Training Trends')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Overdue Training Analysis

In [ ]:
overdue_df = df[df['is_overdue'] == True].copy()
reference_date = pd.Timestamp('2026-03-31')
overdue_df['days_overdue'] = (reference_date - overdue_df['due_date']).dt.days

print(f'Total overdue trainings: {len(overdue_df)}')
print(f'Employees affected: {overdue_df["employee_id"].nunique()}')
print(f'\nOverdue by department:')
print(overdue_df['department'].value_counts())
print(f'\nOverdue by course:')
print(overdue_df['course_name'].value_counts())

In [ ]:
# Top overdue employees
top_overdue = overdue_df.groupby(['employee_id', 'employee_name', 'department']).agg(
    overdue_courses=('course_name', 'count'),
    max_days_overdue=('days_overdue', 'max')
).reset_index().sort_values('overdue_courses', ascending=False)

print('Top 10 employees with most overdue trainings:')
top_overdue.head(10)

## 6. Heatmap: Department × Course

In [ ]:
matrix = df.pivot_table(
    values='status', index='department', columns='course_name',
    aggfunc=lambda x: round((x == 'Completed').sum() / len(x) * 100, 1)
).fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(matrix, annot=True, fmt='.0f', cmap='RdYlGn', linewidths=2,
            linecolor='white', ax=ax, vmin=0, vmax=100)
ax.set_title('Completion Rate: Department × Course (%)')
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

## 7. Key Insights Summary

Based on the analysis above:

1. **Overall completion rate** is ~70%, leaving room for improvement
2. **HR department** leads in completion, while **Marketing** and **Sales** lag behind
3. **Mandatory courses** (Security, Safety, Anti-Harassment) have ~80%+ completion vs ~64% for electives
4. **30%+ of pending trainings** are overdue — urgent action needed
5. Training assignments are **evenly distributed** across months (~450/month)

### Recommendations:
- Send automated overdue reminders
- Investigate why Sales & Marketing departments have lower completion
- Consider making key skill courses mandatory
- Recognize HR department for best performance